# Ikinyamakuru on Colab

Ikinyamakuru files a Kinyarwanda news article under one of the 14 KinNews sections and leaves it for an editor when the score is under the cutoff chosen on development data.

Part 1 runs the desk with a temporary public link, and a CPU runtime is enough. Part 2 links the five notebooks that prepare the data, train the models, and evaluate them.

The [Hugging Face Space](https://huggingface.co/spaces/Gkea/Ikinyamakuru) is always on. The code and documentation are at [github.com/Keaane/Ikinyamakuru](https://github.com/Keaane/Ikinyamakuru).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Keaane/Ikinyamakuru/blob/main/notebooks/colab_app.ipynb)

In [ ]:
!test -d /content/Ikinyamakuru || git clone -q https://github.com/Keaane/Ikinyamakuru.git /content/Ikinyamakuru
%cd /content/Ikinyamakuru
!pip -q install -r app/requirements.txt

## 1. Run the desk

The next cell starts the Flask desk (`app/app.py`) with the committed model and opens a public address through a Cloudflare quick tunnel, which needs no account. The link works while the runtime runs, and the stop cell after it closes the link.

If the tunnel is blocked, `from google.colab import output; output.serve_kernel_port_as_window(7860)` opens the desk in your own browser only.

In [ ]:
import os, pathlib, re, subprocess, time, urllib.request

env = dict(os.environ, PYTHONPATH="src", HOST="127.0.0.1", PORT="7860")
desk = subprocess.Popen(["python", "app/app.py"], env=env,
                        stdout=open("desk.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:7860/health", timeout=1)
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("The desk did not start; see desk.log")

if not pathlib.Path("cloudflared").exists():
    subprocess.run(["wget", "-q", "-O", "cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    os.chmod("cloudflared", 0o755)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", pathlib.Path("tunnel.log").read_text())
    if found:
        url = found.group()
        break
    time.sleep(1)
print(f"Desk: {url}" if url else "No public link yet; see tunnel.log")

In [ ]:
# Stop the desk and close the public link.
for process in (desk, tunnel):
    process.terminate()

## 2. Train and evaluate

The training happens in five notebooks. Each has its own setup cell and opens in Colab; use a GPU runtime (Runtime, Change runtime type, T4 GPU) for notebook 03.

| Notebook | What it does |
| --- | --- |
| [01 Data preparation](https://colab.research.google.com/github/Keaane/Ikinyamakuru/blob/main/notebooks/01_data_preparation.ipynb) | Deduplicates the KinNews CSVs and holds out development |
| [02 Baselines](https://colab.research.google.com/github/Keaane/Ikinyamakuru/blob/main/notebooks/02_baselines.ipynb) | Majority, word and character TF-IDF, and the class-balanced character model |
| [03 AfroXLMR-mini](https://colab.research.google.com/github/Keaane/Ikinyamakuru/blob/main/notebooks/03_afroxlmr.ipynb) | The probe, the full fine-tune, and the class-weighted fine-tune |
| [04 Evaluation](https://colab.research.google.com/github/Keaane/Ikinyamakuru/blob/main/notebooks/04_evaluation.ipynb) | Chooses the system and cutoff on development, then scores the test split once |
| [05 Error analysis](https://colab.research.google.com/github/Keaane/Ikinyamakuru/blob/main/notebooks/05_error_analysis.ipynb) | Figures, confusions, confident mistakes, and the desk examples |

After retraining, run part 1 again to serve the new desk.